# Part 1 — Read an external data source

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
from pathlib import Path


In [11]:
import sys

print(sys.version)

3.14.2 (main, Sep 10 2026, 12:33:04) [GCC 13.3.0]


## Reading the data using a relative path


In [8]:
ROOT = Path.cwd()

if not (ROOT / "data").exists():
    ROOT = ROOT.parent

claims = pd.read_csv(ROOT / "data" / "AutoBi_original.csv")

Dataset: Automobile Bodily Injury Claims (AutoBI)

Source: Insurance Research Council (IRC), 2002.

The dataset contains 1,340 automobile bodily injury claims with information on claimant characteristics, attorney involvement, and economic losses. Economic loss is recorded in thousands of US dollars.

A complete description of the variables, coding schemes, and missing values is available in data/data_dictionary.md.

## Inspecting the data

In [13]:
print("Dimensions:", claims.shape)

print("\nColumns:")
print(claims.columns.tolist())

print("\nIndex:")
print(claims.index)

print("\nData types:")
print(claims.dtypes)

print("\nMissing values:")
print(claims.isna().sum())

print(claims.head())
print(claims.tail())

print("\nInfo:")
print(claims.info())

Dimensions: (1340, 8)

Columns:
['CASENUM', 'ATTORNEY', 'CLMSEX', 'MARITAL', 'CLMINSUR', 'SEATBELT', 'CLMAGE', 'LOSS']

Index:
RangeIndex(start=0, stop=1340, step=1)

Data types:
CASENUM       int64
ATTORNEY      int64
CLMSEX      float64
MARITAL     float64
CLMINSUR    float64
SEATBELT    float64
CLMAGE      float64
LOSS        float64
dtype: object

Missing values:
CASENUM       0
ATTORNEY      0
CLMSEX       12
MARITAL      16
CLMINSUR     41
SEATBELT     48
CLMAGE      189
LOSS          0
dtype: int64
   CASENUM  ATTORNEY  CLMSEX  MARITAL  CLMINSUR  SEATBELT  CLMAGE    LOSS
0        5         1     1.0      NaN       2.0       1.0    50.0  34.940
1       13         2     2.0      2.0       1.0       1.0    28.0  10.892
2       66         2     1.0      2.0       2.0       1.0     5.0   0.330
3       71         1     1.0      1.0       2.0       2.0    32.0  11.037
4       96         2     1.0      4.0       2.0       1.0    30.0   0.138
      CASENUM  ATTORNEY  CLMSEX  MARITAL  CLM

## Checking categorical codes

In [16]:
categorical_vars = [
    "ATTORNEY", "CLMSEX", "MARITAL",
    "CLMINSUR", "SEATBELT"
]

for col in categorical_vars:
    print(f"\n{col}")
    print(claims[col].value_counts(dropna=False).sort_index())


ATTORNEY
ATTORNEY
1    685
2    655
Name: count, dtype: int64

CLMSEX
CLMSEX
1.0    586
2.0    742
NaN     12
Name: count, dtype: int64

MARITAL
MARITAL
1.0    624
2.0    650
3.0     15
4.0     35
NaN     16
Name: count, dtype: int64

CLMINSUR
CLMINSUR
1.0     120
2.0    1179
NaN      41
Name: count, dtype: int64

SEATBELT
SEATBELT
1.0    1270
2.0      22
NaN      48
Name: count, dtype: int64


## Checking for duplicates and nonpositive losses

In [17]:
print("\nDuplicate case numbers:",
      claims["CASENUM"].duplicated().sum())

print("Completely duplicated rows:",
      claims.duplicated().sum())

print("Nonpositive losses:",
      (claims["LOSS"] <= 0).sum())


Duplicate case numbers: 0
Completely duplicated rows: 0
Nonpositive losses: 0


# Modular cleaning

In [18]:
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.cleaning import (
    recode_categories,
    filter_complete,
    summarize_losses
)

## Recoding the original data

In [19]:
claims_clean = recode_categories(claims)

## Summarizing losses by attorney involvement

In [20]:
attorney_summary = summarize_losses(
    claims_clean, "ATTORNEY"
)

display(attorney_summary)

,ATTORNEY,count,mean,median,std
0,No,655,1.864745,0.986,3.894149
1,Yes,685,9.863109,3.417,45.865535


## Preparing observations for the age-loss analysis

In [21]:
age_claims = filter_complete(
    claims_clean, ["CLMAGE", "LOSS"]
)

print("Records available for age analysis:", len(age_claims))

Records available for age analysis: 1151
